In [1]:
import pandas as pd
import numpy as np

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.append(str(PROJECT_ROOT))


In [3]:
# print(sys.path)

In [4]:
# IMPORTING DEFINED FUNCTIONS

from src.text_cleaning import (
    is_zero,
    standardise_columns,
    normalize_text_column,
    normalize_indian_state_names
)

from src.date_utils import (
    extract_date_from_year_and_month
)

from src.geo_utils import (
    map_state_codes
)


In [5]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120) 

In [6]:
DATA_PATH = "../data/raw/"
ADDITIONAL_DATA_PATH = "../data/raw/additional_data/" 

In [7]:
df_main = pd.read_csv(DATA_PATH+"Automated & Unautomated Allocation & Distribution (Rice & Wheat).csv")

In [8]:
df_main.shape

(67522, 9)

In [9]:
print(df_main.columns)

Index(['Country', 'State', 'Year', 'Month', 'District Food Supply Office Implementing This Scheme', 'Commodity',
       'Allocated Quantity (UOM:MT(MetricTonne)), Scaling Factor:1',
       'Allocated Quantity Using Electronic Point Of Sale (Epos) (UOM:MT(MetricTonne)), Scaling Factor:1',
       'Allocated Quantity Without Using Electronic Point Of Sale (Epos) (UOM:MT(MetricTonne)), Scaling Factor:1'],
      dtype='object')


In [10]:
df_main.sample(5)

,Country,State,Year,Month,District Food Supply Office Implementing This Scheme,Commodity,"Allocated Quantity (UOM:MT(MetricTonne)), Scaling Factor:1","Allocated Quantity Using Electronic Point Of Sale (Epos) (UOM:MT(MetricTonne)), Scaling Factor:1","Allocated Quantity Without Using Electronic Point Of Sale (Epos) (UOM:MT(MetricTonne)), Scaling Factor:1"
28231,India,Maharashtra,"Calendar Year (Jan - Dec), 2019","December, 2019","COLLECTOR OFFICE (BRANCH SUPPLY),AKOLA",Wheat,7508.80,3217.97,17.66
11672,India,Madhya Pradesh,"Calendar Year (Jan - Dec), 2020","December, 2020","DISTRICT FOOD OFFICE, SIDHI",Rice,344.80,NaN,21.52
66060,India,Uttar Pradesh,"Calendar Year (Jan - Dec), 2017","November, 2017",FARRUKHABAD,Wheat,4103.39,669.97,NaN
21886,India,Madhya Pradesh,"Calendar Year (Jan - Dec), 2020","April, 2020",Jhabua,Rice,1248.94,1123.91,NaN
64178,India,Tamil Nadu,"Calendar Year (Jan - Dec), 2018","January, 2018",KANCHIPURAM,Wheat,285.56,135.78,NaN


In [11]:
df_main=standardise_columns(df_main)

In [12]:
print(df_main.columns)

Index(['country', 'state', 'year', 'month', 'district_food_supply_office_implementing_this_scheme', 'commodity',
       'allocated_quantity_uommtmetrictonne_scaling_factor1',
       'allocated_quantity_using_electronic_point_of_sale_epos_uommtmetrictonne_scaling_factor1',
       'allocated_quantity_without_using_electronic_point_of_sale_epos_uommtmetrictonne_scaling_factor1'],
      dtype='object')


In [13]:
# RENAMING COLUMNS
df_main = df_main.rename(columns={
    "year": "year_raw",
    "month": "month_raw",
    "district_food_supply_office_implementing_this_scheme": "dfso",
    "allocated_quantity_uommtmetrictonne_scaling_factor1": "total_allocated_qty",
    "allocated_quantity_using_electronic_point_of_sale_epos_uommtmetrictonne_scaling_factor1": "epos_allocated_qty",
    "allocated_quantity_without_using_electronic_point_of_sale_epos_uommtmetrictonne_scaling_factor1": "not_epos_allocated_qty"
})

In [14]:
print(df_main.columns)

Index(['country', 'state', 'year_raw', 'month_raw', 'dfso', 'commodity', 'total_allocated_qty', 'epos_allocated_qty',
       'not_epos_allocated_qty'],
      dtype='object')


In [15]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67522 entries, 0 to 67521
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   country                 67522 non-null  object 
 1   state                   67522 non-null  object 
 2   year_raw                67522 non-null  object 
 3   month_raw               67522 non-null  object 
 4   dfso                    67522 non-null  object 
 5   commodity               67522 non-null  object 
 6   total_allocated_qty     67126 non-null  float64
 7   epos_allocated_qty      45145 non-null  float64
 8   not_epos_allocated_qty  22769 non-null  float64
dtypes: float64(3), object(6)
memory usage: 4.6+ MB


In [16]:
df_main.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
country,67522,1,India,67522,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,67522,35,Uttar Pradesh,11806,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year_raw,67522,5,"Calendar Year (Jan - Dec), 2019",18716,NaN,NaN,NaN,NaN,NaN,NaN,NaN
month_raw,67522,58,"April, 2018",1665,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dfso,67522,1784,HAMIRPUR,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
commodity,67522,2,Rice,37605,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_allocated_qty,67126.0,NaN,NaN,NaN,3999.657775,10485.358078,0.01,849.2175,2889.155,5565.0,839904.3
epos_allocated_qty,45145.0,NaN,NaN,NaN,3120.260046,3022.161033,0.01,724.41,2349.41,4643.67,49492.99
not_epos_allocated_qty,22769.0,NaN,NaN,NaN,1888.189055,9658.131083,0.01,72.2,421.4,2103.96,655202.0


In [17]:
print(df_main.duplicated().sum())

0


In [18]:
print(df_main.isna().sum())

country                       0
state                         0
year_raw                      0
month_raw                     0
dfso                          0
commodity                     0
total_allocated_qty         396
epos_allocated_qty        22377
not_epos_allocated_qty    44753
dtype: int64


In [19]:
df_main["country"].value_counts()

country
India    67522
Name: count, dtype: int64

In all records, the country is India (as this is PDS Data of India). So, country column is irrelevant for visualisation or modeling (it is already understood to be India)

In [20]:
df_main.drop(columns=["country"], inplace=True)

In [21]:
# STANDARDISING TEXT COLUMNS
# remove whitespace, turn to lowercase
df_main["state"] = normalize_text_column(df_main["state"])
df_main["commodity"] = normalize_text_column(df_main["commodity"])


In [22]:
df_main["state"].unique()

array(['tamil nadu', 'andhra pradesh', 'arunachal pradesh',
       'chhattisgarh', 'daman & diu', 'gujarat', 'haryana',
       'himachal pradesh', 'jammu and kashmir', 'jharkhand', 'karnataka',
       'kerala', 'ladakh', 'madhya pradesh', 'maharashtra', 'manipur',
       'meghalaya', 'mizoram', 'nagaland', 'rajasthan', 'telangana',
       'tripura', 'uttar pradesh', 'uttarakhand', 'west bengal', 'assam',
       'bihar', 'delhi', 'goa', 'odisha', 'punjab', 'sikkim',
       'andaman and nicobar islands', 'lakshadweep',
       'dadra & nagar haveli'], dtype=object)

In [23]:
# SPECIFIC STATE  NORMALISATION 
# Change "daman & diu" and "dadra & nagar haveli" to one state - "dadra and nagar haveli and daman and diu"
df_main["state"] = normalize_indian_state_names(df_main["state"])

In [24]:
# MAPPING STATE CODES
df_main["state_code"] = map_state_codes(df_main, state_col="state")

In [25]:
df_main.head()

,state,year_raw,month_raw,dfso,commodity,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty,state_code
0,tamil nadu,"Calendar Year (Jan - Dec), 2021","October, 2021",Pudukkottai,rice,6445.00,5595.02,NaN,TN
1,tamil nadu,"Calendar Year (Jan - Dec), 2021","October, 2021",Pudukkottai,wheat,599.00,395.49,NaN,TN
2,andhra pradesh,"Calendar Year (Jan - Dec), 2021","September, 2021",Anantapur,rice,13030.41,12247.06,NaN,AP
3,andhra pradesh,"Calendar Year (Jan - Dec), 2021","September, 2021",Chittoor,rice,14176.89,13199.87,NaN,AP
4,andhra pradesh,"Calendar Year (Jan - Dec), 2021","September, 2021",East Godavari,rice,16541.31,15778.81,NaN,AP


In [26]:
print("Missing state_code:", df_main["state_code"].isna().sum())

Missing state_code: 0


In [27]:
# EXTRACT YEAR AND MONTH FROM THE COLUMNS, ADD QUARTER COLUMN
df_main['date'] = extract_date_from_year_and_month(df_main['year_raw'],df_main['month_raw'])

In [28]:
df_main.columns

Index(['state', 'year_raw', 'month_raw', 'dfso', 'commodity', 'total_allocated_qty', 'epos_allocated_qty',
       'not_epos_allocated_qty', 'state_code', 'date'],
      dtype='object')

In [29]:
df_main.sample(5)

,state,year_raw,month_raw,dfso,commodity,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty,state_code,date
50800,west bengal,"Calendar Year (Jan - Dec), 2018","October, 2018",DCFS-Purulia,wheat,6761.61,NaN,6736.80,WB,2018-10-31
47550,uttar pradesh,"Calendar Year (Jan - Dec), 2018","December, 2018",VARANASI,rice,5552.92,5052.65,NaN,UP,2018-12-31
32150,bihar,"Calendar Year (Jan - Dec), 2019","September, 2019",DISTRICT SUPPY OFFICE KHAGARIA,wheat,3046.32,NaN,3046.32,BR,2019-09-30
26079,chhattisgarh,"Calendar Year (Jan - Dec), 2020","January, 2020",Kanker,rice,25.99,NaN,25.99,CG,2020-01-31
5552,uttar pradesh,"Calendar Year (Jan - Dec), 2021","May, 2021",Pilibhit,wheat,5134.45,4922.57,NaN,UP,2021-05-31


In [30]:
df_main.drop(columns=["year_raw","month_raw"], inplace=True) 
# as month and year data already in "year" and "month" column

In [31]:
df_main.columns

Index(['state', 'dfso', 'commodity', 'total_allocated_qty', 'epos_allocated_qty', 'not_epos_allocated_qty',
       'state_code', 'date'],
      dtype='object')

In [32]:
# UPON REMOVING DISTRICT-LEVEL INFO AND CHECKING DUPLICTAES, WE FOUND 46 DUPLICATED() ROWS
# HOWEVER, THERE WERE NO DATA DUPLICATED IN ORIGINAL DATAFRAME
# SO, RE-CHECKING
# EXACT DUPES - IF ALL VALUES ARE SAME
exact_dupes = df_main[df_main.duplicated(keep=False)]
exact_dupes.sort_values(["state","commodity","date"])

,state,dfso,commodity,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty,state_code,date


In [33]:
# NO ROW IS ENTIRELY DUPLICATED
# CHECK IF REMOVING DISTRICT-LEVEL GRANULARITY CREATES DUPLICATES, IF YES, INSPECT FURTHER

In [34]:
subset_cols = [col for col in df_main.columns if col != "dfso"]

df_main.duplicated(
    subset=subset_cols
).sum()

np.int64(46)

In [35]:
# SAVE THE DUPLICATED VALUES (EXCEPT FOR DISTRICT) TO A DATAFRAME
non_exact_dupes = df_main[
    df_main.duplicated(subset=subset_cols, keep=False)
].copy()

In [36]:
non_exact_dupes = non_exact_dupes.sort_values(
    ["state","commodity","date"]
)

In [37]:
unique_states = sorted(non_exact_dupes["state"].unique())
unique_states

['andhra pradesh',
 'arunachal pradesh',
 'assam',
 'chhattisgarh',
 'himachal pradesh',
 'karnataka',
 'madhya pradesh',
 'tamil nadu',
 'telangana',
 'uttar pradesh']

In [38]:
for state in unique_states:

    state_df = non_exact_dupes[non_exact_dupes["state"] == state]

    print("\nSTATE:", state)

    for keys, group in state_df.groupby(
        ["commodity","date"]
    ):
        print("\nGroup:", keys)
        display(group[["dfso","total_allocated_qty",
                       "epos_allocated_qty",
                       "not_epos_allocated_qty"]])


STATE: andhra pradesh

Group: ('rice', Timestamp('2019-01-31 00:00:00'))


,dfso,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
44463,DSO OFFICE PRAKASAM,92.96,NaN,92.96
44467,DSO OFFICE WEST GODAVARI,92.96,NaN,92.96



Group: ('rice', Timestamp('2019-02-28 00:00:00'))


,dfso,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
42904,DSO OFFICE CHITTOOR,5.86,NaN,5.86
42907,DSO OFFICE GUNTUR,5.86,NaN,5.86



STATE: arunachal pradesh

Group: ('rice', Timestamp('2018-07-31 00:00:00'))


...

,dfso,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
67469,MEDAK,0.01,NaN,NaN
67477,SIDDIPET,0.01,NaN,NaN
67479,VIKARABAD,0.01,NaN,NaN



Group: ('wheat', Timestamp('2017-09-30 00:00:00'))


,dfso,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
67223,JAGITYAL,0.03,NaN,NaN
67225,JANAGOAN,0.01,NaN,NaN
67229,MAHBUBNAGAR,0.03,NaN,NaN
67231,MEDAK,0.01,NaN,NaN
67241,SIDDIPET,0.01,NaN,NaN
67243,VIKARABAD,0.01,NaN,NaN
67247,WARANGAL RURAL,0.01,NaN,NaN



STATE: uttar pradesh

Group: ('rice', Timestamp('2021-05-31 00:00:00'))


,dfso,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
5425,Chandauli,3383.34,NaN,NaN
5435,DSO Chandauli,3383.34,NaN,NaN


In [39]:
### A SUMMARY OF THE DUPLICATED DISTRICT ANALYSIS - BEFORE DROPPING DFSO COLUMN

# ----------------------
# ANDHRA PRADESH
# ----------------------
# 1. Chittor & Guntur - separate districts - include both
# 2. Prakasam & West Godavari - separate districts - include both

# -------------------
# ARUNACHAL PRADESH
# -------------------
# 1. Anom Apang => modified to Shi Yomi district - include; Tagom Ketan => no such district - delete
# 2. Peken Yomcha => no such district - delete; Upper Subansiri - exists - include

# -------------------
# ASSAM
# -------------------
# Barpeta - exist, distinct - include
# Dibrugarh - exist, distinct - include
# Dhemaji - exist, distinct - include
# Hailakandi - exist, distinct - include
# Sonitpur - exist, distinct - include
# Morigaon - exist, distinct - include

# -------------------
# CHATTISGARH
# -------------------
# Bijapur - exist, distinct, include
# Sukma - exist, distinct, include

# -------------------
# HIMACHAL PRADESH
# -------------------
# BILASPUR	- exist, distinct, include
# HAMIRPUR	- exist, distinct, include
# KANGRA	- exist, distinct, include
# KULLU	- exist, distinct, include
# MANDI	- exist, distinct, include
# SHIMLA	- exist, distinct, include
# SIRMAUR	- exist, distinct, include
# SOLAN	- exist, distinct, include
# UNA	- exist, distinct, include
# Kullu,District Controller, FCS and CA => rename as KULLU, include
# Sirmour,District Controller, FCS and CA => rename as Sirmaur, include


# -------------------
# KARNATAKA
# -------------------
# Ballari - exist, distinct - include
# Raichur - exist, distinct, include


# -------------------
# MADHYA PRADESH
# -------------------
# 66958	DSO ASHOKNAGAR	- exists, distinct - include
# 66998	DSO JHABUA	- exists, distinct - include
# 67036	DSO SHAJAPUR	- exists, distinct - include
# 67046	DSO TIKAMGARH	- exists, distinct - include


# -------------------
# TAMIL NADU
# -------------------
# ARIYALUR	- exists, distinct - include
# CUDDALORE	- exists, distinct - include
# DHARMAPURI	- exists, distinct - include
# DINDIGUL	- exists, distinct - include
# KANCHIPURAM	- exists, distinct - include
# NILGIRIS	- exists, distinct - include
# RAMANATHAPURAM	- exists, distinct - include
# SIVAGANGAI	- exists, distinct - include
# THIRUVALLUR	- exists, distinct - include
# THIRUVARUR	- exists, distinct - include

# -------------------
# TELANGANA
# -------------------
# Adilabad	 - exists, distinct - include
# Khammam   - exists, distinct - include
# Nalgonda  - exists, distinct - include
# Ranga Reddy   - exists, distinct - include
# MEDAK	    - exists, distinct - include
# SIDDIPET  - exists, distinct - include
# VIKARABAD - exists, distinct - include
# JAGITYAL  - exists, distinct - include
# JANAGOAN  - exists, distinct - include
# MAHBUBNAGAR   - exists, distinct - include
# MEDAK - exists, distinct - include
# SIDDIPET  - exists, distinct - include
# VIKARABAD - exists, distinct - include
# WARANGAL RURAL    - exists, distinct - include	

# -------------------
# UTTAR PRADESH
# -------------------
# Chandauli	- exists, distinct - include
# DSO Chandauli - copy of chandauli - remove


In [40]:
# cleaning the repeated and erroneous values
rows_to_delete = [52517, 54162, 44505, 5435]
df_main = df_main.drop(index=rows_to_delete)

In [41]:
df_main.index.isin(rows_to_delete).sum()

np.int64(0)

In [42]:
df_main = df_main.reset_index(drop=True)

In [43]:
# As district-level modeling is beyond the scope of this project, we can drop DFSO column too. 
df_main.drop(columns=["dfso"],inplace=True)
# As state date already encoded in state_code, drop state column too
df_main.drop(columns=["state"],inplace=True)

In [44]:
print(df_main.columns)

Index(['commodity', 'total_allocated_qty', 'epos_allocated_qty', 'not_epos_allocated_qty', 'state_code', 'date'], dtype='object')


In [45]:
# REORDERING COLUMNS
desired_column_order = [
    "state_code",
    "commodity",
    "date",
    "total_allocated_qty",
    "epos_allocated_qty",
    "not_epos_allocated_qty"
]
existing_cols = [c for c in desired_column_order if c in df_main.columns]
df_main = df_main[existing_cols]

In [46]:
df_main.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
state_code,67518,34,UP,11805,NaN,NaN,NaN,NaN,NaN,NaN,NaN
commodity,67518,2,rice,37601,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,67518,NaN,NaN,NaN,2019-09-24 22:08:40.181284608,2017-01-31 00:00:00,2018-10-31 00:00:00,2019-08-31 00:00:00,2020-08-31 00:00:00,2021-10-31 00:00:00,NaN
total_allocated_qty,67122.0,NaN,NaN,NaN,3999.840912,0.01,849.255,2889.17,5565.0,839904.3,10485.6379
epos_allocated_qty,45145.0,NaN,NaN,NaN,3120.260046,0.01,724.41,2349.41,4643.67,49492.99,3022.161033
not_epos_allocated_qty,22769.0,NaN,NaN,NaN,1888.189055,0.01,72.2,421.4,2103.96,655202.0,9658.131083


In [47]:
df_main.shape

(67518, 6)

In [48]:
df_main.dtypes

state_code                        object
commodity                         object
date                      datetime64[ns]
total_allocated_qty              float64
epos_allocated_qty               float64
not_epos_allocated_qty           float64
dtype: object

In [49]:
df_main.sample(5)

,state_code,commodity,date,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
30092,TN,wheat,2019-11-30,445.09,406.58,NaN
46699,MP,rice,2018-12-31,290.70,NaN,284.23
56568,MN,rice,2018-06-30,2057.67,NaN,2057.67
13187,MH,rice,2020-11-30,4240.13,4056.86,NaN
35812,MH,rice,2019-07-31,4372.72,1807.07,123.24


In [50]:
df_main.isna().sum()

state_code                    0
commodity                     0
date                          0
total_allocated_qty         396
epos_allocated_qty        22373
not_epos_allocated_qty    44749
dtype: int64

In [51]:
df_main["total_allocated_qty"].isna().sum()

np.int64(396)

In [52]:
alloc_cols = [
    "total_allocated_qty",
    "epos_allocated_qty",
    "not_epos_allocated_qty"
]

In [53]:
# convert to numeric and coerce invalid values to NaN
df_main[alloc_cols] = df_main[alloc_cols].apply(
    pd.to_numeric, errors="coerce"
)

In [54]:
df_main["total_allocated_qty"] = df_main["total_allocated_qty"].fillna(0)
df_main["epos_allocated_qty"] = df_main["epos_allocated_qty"].fillna(0)
df_main["not_epos_allocated_qty"] = df_main["not_epos_allocated_qty"].fillna(0)

In [55]:
# validation
assert (df_main[alloc_cols] >= 0).all().all()

In [56]:
df_main["total_allocated_qty"].isna().sum()

np.int64(0)

In [57]:
df_main.isna().sum()

state_code                0
commodity                 0
date                      0
total_allocated_qty       0
epos_allocated_qty        0
not_epos_allocated_qty    0
dtype: int64

In [58]:
# Identify rows where total allocation is inconsistent
invalid = (
    (df_main["total_allocated_qty"] < df_main["epos_allocated_qty"]) |
    (df_main["total_allocated_qty"] < df_main["not_epos_allocated_qty"])
)

inconsistent_alloc = df_main[invalid].copy()

print("Number of inconsistent rows:", inconsistent_alloc.shape[0])

Number of inconsistent rows: 3706


In [59]:
inconsistent_alloc.sample(10)

,state_code,commodity,date,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
64142,RJ,wheat,2018-01-31,2100.06,2102.89,0.00
35974,RJ,wheat,2019-07-31,2136.94,2607.39,0.00
9421,UP,wheat,2021-02-28,1587.49,3201.85,0.00
43917,RJ,wheat,2019-02-28,43.70,0.00,343.98
22173,RJ,wheat,2020-04-30,2587.98,2649.35,0.00
30992,JH,wheat,2019-10-31,1314.09,1336.33,0.00
35749,MH,wheat,2019-07-31,3961.37,4003.34,0.00
66842,HR,wheat,2017-09-30,0.36,1.30,0.00
62287,HR,wheat,2018-02-28,1990.89,3176.28,0.00
56261,KL,wheat,2018-06-30,726.65,760.78,0.00


In [60]:
# Fix cases where EPOS is zero → use non-EPOS
df_main.loc[
    invalid & (df_main["epos_allocated_qty"] == 0),
    "total_allocated_qty"
] = df_main["not_epos_allocated_qty"]

# 3. Fix cases where non-EPOS is zero → use EPOS
df_main.loc[
    invalid & (df_main["not_epos_allocated_qty"] == 0),
    "total_allocated_qty"
] = df_main["epos_allocated_qty"]

In [61]:
# Recompute invalid rows after first correction
invalid_after = (
    (df_main["total_allocated_qty"] < df_main["epos_allocated_qty"]) |
    (df_main["total_allocated_qty"] < df_main["not_epos_allocated_qty"])
)

inconsistent_alloc = df_main[invalid_after].copy()

print("Number of inconsistent rows:", inconsistent_alloc.shape[0])

Number of inconsistent rows: 31


In [62]:
inconsistent_alloc.head(15)

,state_code,commodity,date,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
3944,RJ,wheat,2021-06-30,10576.28,10965.22,51.56
4268,UK,wheat,2021-06-30,1525.39,233.85,1930.53
5233,RJ,wheat,2021-05-31,8793.89,9131.40,212.00
14785,RJ,wheat,2020-10-31,4502.37,4508.52,33.74
18284,WB,rice,2020-07-31,3334.58,4578.99,82.37
18310,WB,rice,2020-07-31,1980.46,3913.30,90.64
19734,WB,rice,2020-06-30,3334.57,4591.19,40.80
19744,WB,rice,2020-06-30,6942.43,8802.41,66.48
19760,WB,rice,2020-06-30,1980.96,4136.31,75.58
21144,WB,rice,2020-05-31,3334.56,4531.36,102.77


In [63]:
inconsistent_alloc.tail(16)

,state_code,commodity,date,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
28899,WB,rice,2019-12-31,4110.73,5365.74,179.29
32758,MH,rice,2019-09-30,4963.00,4974.25,0.02
32759,MH,wheat,2019-09-30,7321.17,7327.40,0.02
38095,UK,rice,2019-06-30,1146.39,35.28,1299.01
38096,UK,wheat,2019-06-30,726.22,21.95,837.70
38975,MH,wheat,2019-05-31,1137.22,1176.40,134.00
56478,MH,rice,2018-06-30,2311.06,2590.63,26.84
56479,MH,wheat,2018-06-30,2980.24,3348.54,38.51
56489,MH,wheat,2018-06-30,9311.71,4900.45,16731.52
56500,MH,rice,2018-06-30,1574.61,2722.12,937.16


In [64]:
# Fix remaining inconsistencies using max(epos, not_epos)
df_main.loc[
    invalid_after,
    "total_allocated_qty"
] = df_main.loc[
    invalid_after,
    ["epos_allocated_qty", "not_epos_allocated_qty"]
].max(axis=1)

In [65]:
# Final integrity check
check_final = (
    (df_main["total_allocated_qty"] < df_main["epos_allocated_qty"]) |
    (df_main["total_allocated_qty"] < df_main["not_epos_allocated_qty"])
)

print("Remaining inconsistent rows:", check_final.sum())

Remaining inconsistent rows: 0


In [66]:
is_zero(df_main).sum()

state_code                    0
commodity                     0
date                          0
total_allocated_qty           0
epos_allocated_qty        22373
not_epos_allocated_qty    44749
dtype: int64

In [67]:
df_main.drop(columns=["epos_allocated_qty","not_epos_allocated_qty"],inplace=True)

In [68]:
df_main.dtypes

state_code                     object
commodity                      object
date                   datetime64[ns]
total_allocated_qty           float64
dtype: object

In [69]:
df_main.isna().sum()

state_code             0
commodity              0
date                   0
total_allocated_qty    0
dtype: int64

In [70]:
df_main.to_csv("../data/cleaned/primary_data_clean.csv", index=False)
